In [128]:
from utils.s3_util import S3Util
import os
import json
import pandas as pd
import numpy as np

In [105]:
s3_util = S3Util()
s3_client = s3_util.get_s3_client()

In [106]:
raw_file_name = os.getenv("RAW_FILE_NAME")
bucket_name = os.getenv("S3_BUCKET_NAME")

In [107]:
fhv_raw_data_file = s3_client.get_object(Bucket=bucket_name, Key=raw_file_name)
fhv_raw_data_content = fhv_raw_data_file['Body'].read().decode('utf-8')
fhv_raw_data_json = json.loads(fhv_raw_data_content)

In [108]:
fhv_raw_data_df = pd.DataFrame(fhv_raw_data_json)
fhv_raw_data_df.head()

,:id,:version,:created_at,:updated_at,active,vehicle_license_number,name,license_type,expiration_date,permit_license_number,...,base_type,veh,base_telephone_number,website,base_address,reason,last_date_updated,last_time_updated,certification_date,hack_up_date
0,row-s5xw-3q3c_ewtu,rv-4bdt-3adw-gks2,2026-03-20T19:17:26.390Z,2026-03-20T19:17:26.390Z,YES,6032728,"UPPAL, ARSHDEEP",For Hire Vehicle,2026-06-30T00:00:00.000,AA005,...,BLACK CAR,HYB,(718) 472-9800,EXITLUXURYINC@GMAIL.COM,29 - 10 36 AVENUE LONGISLAND CITY NY 11106,G,2026-03-20T00:00:00.000,14:00,NaN,NaN
1,row-weff~jwcp-ttmc,rv-gqef.w9ky_25zw,2026-03-20T19:17:26.390Z,2026-03-20T19:17:26.390Z,YES,5953896,"ELIHORI,ASIM,SALIH",For Hire Vehicle,2027-11-07T00:00:00.000,AA006,...,BLACK CAR,NaN,(718) 482-1818,INFO@ZYNC.NYC,12-04 44 AVENUE LIC NY 11101,G,2026-03-20T00:00:00.000,14:00,2019-11-07T00:00:00.000,2019-11-07T00:00:00.000
2,row-vs5n~hqkv~jnws,rv-9wsj~x9vq.rzaf,2026-03-20T19:17:26.390Z,2026-03-20T19:17:26.390Z,YES,5729129,"LORENZO,MARTIN",For Hire Vehicle,2026-10-12T00:00:00.000,AA010,...,LIVERY,NaN,(718) 222-0600,TLC.LIVERY@CURBTRANSPORTATION.COM,11 - 11 34 AVENUE ASTORIA NY 11106,G,2026-03-20T00:00:00.000,14:00,2023-11-17T00:00:00.000,2023-11-20T00:00:00.000
3,row-3bcw~j3dt.ukas,rv-rqiy.duzz-tdrv,2026-03-20T19:17:26.390Z,2026-03-20T19:17:26.390Z,YES,6035913,"GULYAMOV,,AMINJON",For Hire Vehicle,2026-06-30T00:00:00.000,AA013,...,BLACK CAR,HYB,(212) 741-1562,TLC@NYC2WAY.COM,241 37 STREET BROOKLYN NY 11232,G,2026-03-20T00:00:00.000,14:00,NaN,NaN
4,row-5692_ewt9.etit,rv-hpj6-ff6t~9czw,2026-03-20T19:17:26.390Z,2026-03-20T19:17:26.390Z,YES,6033635,"KHAN,DEWAN,M",For Hire Vehicle,2026-06-30T00:00:00.000,AA017,...,BLACK CAR,HYB,(347) 590-7005,SOUTHBRONX250@GMAIL.COM,250 BROOK AVENUE BRONX NY 10454,G,2026-03-20T00:00:00.000,14:00,NaN,NaN


In [109]:
fhv_raw_data_json[0]

{':id': 'row-s5xw-3q3c_ewtu',
 ':version': 'rv-4bdt-3adw-gks2',
 ':created_at': '2026-03-20T19:17:26.390Z',
 ':updated_at': '2026-03-20T19:17:26.390Z',
 'active': 'YES',
 'vehicle_license_number': '6032728',
 'name': 'UPPAL, ARSHDEEP',
 'license_type': 'For Hire Vehicle',
 'expiration_date': '2026-06-30T00:00:00.000',
 'permit_license_number': 'AA005',
 'dmv_license_plate_number': 'T117661C',
 'vehicle_vin_number': 'KMHLM4AJ2NU023825',
 'wheelchair_accessible': 'PILOT',
 'vehicle_year': '2022',
 'base_number': 'B03152',
 'base_name': 'EXIT LUXURY INC.',
 'base_type': 'BLACK CAR',
 'veh': 'HYB',
 'base_telephone_number': '(718) 472-9800',
 'website': 'EXITLUXURYINC@GMAIL.COM',
 'base_address': '29 - 10   36 AVENUE LONGISLAND CITY NY 11106',
 'reason': 'G',
 'last_date_updated': '2026-03-20T00:00:00.000',
 'last_time_updated': '14:00'}

In [110]:
fhv_raw_data_df.drop(columns=["license_type", "reason", "hack_up_date", "certification_date", ":version"], inplace=True)

In [111]:
fhv_raw_data_df.rename(columns={
    ":id": "id",
    ":created_at": "created_at",
    ":updated_at": "updated_at",
    "veh": "hybrid_vehicle_indicator",
}, inplace=True)

In [115]:
fhv_raw_data_df["wheelchair_accessible"] = fhv_raw_data_df[
    "wheelchair_accessible"
].fillna("NON-WAV")
fhv_raw_data_df["hybrid_vehicle_indicator"] = fhv_raw_data_df[
    "hybrid_vehicle_indicator"
].fillna("NON-HYB")
fhv_raw_data_df["hybrid_vehicle_indicator"] = fhv_raw_data_df[
    "hybrid_vehicle_indicator"
].replace(
    {
        "NON": "NON-HYB",
        "N": "NON-HYB",
        "DSE": "DIESEL",
        "DSEL": "DIESEL",
        "WAV": "NON-HYB",
    }
)

In [113]:
fhv_raw_data_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 104055 entries, 0 to 104054
Data columns (total 21 columns):
 #   Column                    Non-Null Count   Dtype
---  ------                    --------------   -----
 0   id                        104055 non-null  str  
 1   created_at                104055 non-null  str  
 2   updated_at                104055 non-null  str  
 3   active                    104055 non-null  str  
 4   vehicle_license_number    104055 non-null  str  
 5   name                      104055 non-null  str  
 6   expiration_date           104055 non-null  str  
 7   permit_license_number     1091 non-null    str  
 8   dmv_license_plate_number  104053 non-null  str  
 9   vehicle_vin_number        104052 non-null  str  
 10  wheelchair_accessible     104055 non-null  str  
 11  vehicle_year              104052 non-null  str  
 12  base_number               104055 non-null  str  
 13  base_name                 104055 non-null  str  
 14  base_type                 10405

In [120]:
fhv_raw_data_df['wheelchair_accessible'].value_counts()

wheelchair_accessible
NON-WAV    95807
WAV         7809
PILOT        439
Name: count, dtype: int64

In [130]:
fhv_raw_data_df['sequence_number'] = np.arange(len(fhv_raw_data_df))

In [132]:
fhv_raw_data_df['partition_id'] = fhv_raw_data_df['sequence_number'] % 15

In [133]:
fhv_raw_data_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 104055 entries, 0 to 104054
Data columns (total 23 columns):
 #   Column                    Non-Null Count   Dtype
---  ------                    --------------   -----
 0   id                        104055 non-null  str  
 1   created_at                104055 non-null  str  
 2   updated_at                104055 non-null  str  
 3   active                    104055 non-null  str  
 4   vehicle_license_number    104055 non-null  str  
 5   name                      104055 non-null  str  
 6   expiration_date           104055 non-null  str  
 7   permit_license_number     1091 non-null    str  
 8   dmv_license_plate_number  104053 non-null  str  
 9   vehicle_vin_number        104052 non-null  str  
 10  wheelchair_accessible     104055 non-null  str  
 11  vehicle_year              104052 non-null  str  
 12  base_number               104055 non-null  str  
 13  base_name                 104055 non-null  str  
 14  base_type                 10405

In [140]:
for pid in fhv_raw_data_df['partition_id'].unique():
    partition_df = fhv_raw_data_df[fhv_raw_data_df['partition_id'] == pid]
    partition_df.to_csv(f'fhv_data_{pid}.csv', index=False)